In [ ]:
import os
import sys

print("현재 작업 디렉토리:", os.getcwd())

# 이 노트북은 repo의 src/ 디렉토리에서 실행하세요 (jupyter notebook을 src/에서 띄우거나 아래 경로를 맞춰주세요)
# os.chdir('src')  # 필요 시 주석 해제


In [ ]:
GPU_NUM = 0
GPU_NUM = str(GPU_NUM)
import os



os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID" 
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_NUM

import shutil
import warnings
import contextlib
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from copy import deepcopy
from IPython.display import clear_output

warnings.filterwarnings(action='ignore')
plt.style.use(plt.style.available[-3])
plt.rcParams['image.cmap'] = 'gray'

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, Dataset, ConcatDataset
from torchvision import transforms

torch.set_default_dtype(torch.float32)

GPU = torch.device('cuda')
CPU = torch.device('cpu')
print(torch.cuda.is_available(), ': ', torch.cuda.get_device_name(0))
torch.cuda.empty_cache()

In [ ]:
from utils.utils import *
from utils.loss import *
from utils.dataset import *
from networks.CBAS import *
from options.hyper_parameters import HP

In [ ]:
context = 0
depth = 2*context + 1
name = "CBAGS"
# name = f'{depth}_nonse_attention_HybridUNet'
# name = 'Origin_AttentionUnet'

In [ ]:
import logging

logging.basicConfig(
    filename=f'{name}.out',
    filemode='w',
    level=logging.INFO,
    format='[%(asctime)s] %(message)s',
    datefmt='%m/%d %H:%M:%S'
)

In [ ]:
hp = HP(model = SuperEnhancedAFMSUNet2D(out_channels=1, context=context), name = name)
hp.__dict__

In [ ]:
dicomDataset = Dicom2dDataset

In [ ]:
tf = transforms.Compose([
    transforms.Lambda(lambda x: (x - x.min()) / (x.max() - x.min() + 1e-8))
])

# repo 루트 기준 data/h5 (src/ 에서 실행 시 상대경로 "../data/h5")
root_dir = "../data/h5"
target_dir = ['5mm 미만', '5mm 이상', 'normal', '추가 병변']


In [ ]:
dataset = dicomDataset(
    root_dir=root_dir,
    target_dir=target_dir,
    transform=tf,
    context=context,
    only_lesion=False
)
if hp.same_per_sample:
    
    selector = NormalSliceSelector(dataset, seed = hp.seed, n_clusters=393)
    normal_subset, selected_indices, metas, sideinfos = selector.run_and_make_subset()

    lesion_dataset = dicomDataset(
        root_dir=root_dir,
        target_dir=target_dir,
        transform=tf,
        context=context,
        only_lesion=True
    )

    dataset = ConcatDataset([lesion_dataset, normal_subset])

print(f"최종 데이터셋 샘플 수: {len(dataset)}")

In [ ]:
collate_fn = custom_collate_fn

In [ ]:
for idx, (fold, train_subset, val_subset, test_subset) in enumerate(k_fold_split(dataset,val_ratio=0.25, k=5, seed=hp.seed)):
    logging.info(f"[Fold {fold}]")
    logging.info(f"  Train: {len(train_subset)}, Val: {len(val_subset)}, Test: {len(test_subset)}")
    
    best_metric = float('inf')  # GUL loss 기준은 낮을수록 좋음

    hp = HP(model = SuperEnhancedAFMSUNet2D(out_channels=1, context=context), name = name)
    model = hp.model
    # hp.position_embedding = False # PE 제외

    optimizer = torch.optim.AdamW(model.parameters(), lr=hp.optimizer_lr, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=hp.scheduler_step, gamma=hp.scheduler_gamma)

    train_loader, valid_loader, test_loader = get_loaders(train_subset, val_subset, test_subset, batch_size=hp.batch_size, collate_fn=collate_fn, worker_init_fn = seed_worker(), g = torch.Generator().manual_seed(hp.seed))
    data_loader = [train_loader, valid_loader]
    visualize_dataloader_batch(train_loader, batch_index=2, sample_index=1)

    if hp.multi_gpu: model = nn.DataParallel(model)
    model = model.to(hp.device)

    torch.cuda.empty_cache()
    logging.info(f'{"Model":<20}: {hp.name}')

    Focal_func = FocalLoss()
    BCE_func = BCE()
    dice_func = DiceLoss()
    gul_func = GeneralUnionLoss()
    # Boundary_func = BoundaryLoss()

    Recall_func = Recall()
    Precision_func = Precision()
    DiceScore_func = DiceScore()
    Specificity_func = Specificity()
    ACC_func = ACC()

    loss_keys = ['Focal','Dice','BCE','GUL']
    rate_keys = ['Precision','Recall','Dice', 'Specificity', 'ACC']

    epoch_s, epoch_e = 1, hp.epochs+1
    scaler = torch.cuda.amp.GradScaler()
    loss_epoch = torch.zeros([epoch_e, 2, len(loss_keys)])
    rate_epoch = torch.zeros([epoch_e, 2, len(rate_keys)])
    if hp.epoch_load!=None:
        last_point = torch.load(f'{hp.path_model}/fold_{idx}/history.pt')
        model.load_state_dict(last_point['model'])
        optimizer.load_state_dict(last_point['optimizer'])
        scaler.load_state_dict(last_point['scaler'])
        scheduler.load_state_dict(last_point['scheduler'])
        epoch_s = last_point['epoch']
        history = last_point['history']
        loss_epoch = history['loss']
        rate_epoch = history['rate']
        loss_epoch[:epoch_s] = history['loss'][:epoch_s]
        rate_epoch[:epoch_s] = history['rate'][:epoch_s]
        logging.info(f'※ Continual Learning: {epoch_s-1} Epoch ※')
        
    for epoch in range(epoch_s, epoch_e):

        for i, phase in enumerate(['Train', 'Valid']):
            if phase=='Train':
                model.train()
                context_manager = contextlib.nullcontext()
            if phase=='Valid':
                model.eval()
                context_manager = torch.no_grad()
            
            loss_batch = torch.zeros([len(data_loader[i]), len(loss_keys)])
            rate_batch = torch.zeros([len(data_loader[i]), len(rate_keys)])
            with context_manager:
                for k, data in enumerate(data_loader[i]):
                    X, y = data[0].to(hp.device), data[1].to(hp.device)

                    if hp.position_embedding:
                        FOV_batch = []
                        for fov_percent in data[3]:
                            tmp = []
                            for fov in fov_percent:
                                tmp.append(fov['FOV_Z_percent'])
                            FOV_batch.append(tmp)
                        FOV_batch = torch.tensor(FOV_batch)
                        pe = FOV_batch.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, X.shape[-2], X.shape[-1]).contiguous().to(hp.device)

                    with torch.cuda.amp.autocast(enabled=False): #enc -> dec overflow : nan
                        if hp.position_embedding:
                            p_main, p_coarse, p_seg4 = model(X, pe)
                        else:
                            p_main, p_coarse, p_seg4 = model(X)

                        # 주석 처리된 기존 loss들
                        Focal_res = Focal_func(p_main, y)
                        Dice_res = dice_func(p_main, y)
                        BCE_res = BCE_func(p_main, y)
                        GUL_res = gul_func(p_main, y)
                        # Boundary_res = Boundary_func(p_main, y)

                        p_coarse_up = F.interpolate(p_coarse, size=y.shape[2:], mode='bilinear', align_corners=False)
                        p_seg4_up = F.interpolate(p_seg4, size=y.shape[2:], mode='bilinear', align_corners=False)

                        Focal_coarse = Focal_func(p_coarse_up, y) * 0.3  # aux 가중치 낮게
                        Dice_coarse = dice_func(p_coarse_up, y) * 0.3
                        GUL_coarse = gul_func(p_coarse_up, y) * 0.3

                        Focal_seg4 = Focal_func(p_seg4_up, y) * 0.2
                        Dice_seg4 = dice_func(p_seg4_up, y) * 0.2

                        loss_batch[k, 0] = Focal_res.detach()
                        loss_batch[k, 1] = Dice_res.detach()
                        loss_batch[k, 2] = BCE_res.detach()
                        loss_batch[k, 3] = GUL_res.detach()
                        # loss_batch[k, 4] = Boundary_res.detach()

                        loss_final = Focal_res * hp.focal_loss_weight + Dice_res * hp.dice_loss_weight + BCE_res*hp.bce_loss_weight + GUL_res*hp.gul_loss_weight + (Focal_coarse + Dice_coarse  + GUL_coarse) * 0.3 + (Dice_seg4 + Focal_seg4)*0.2

                    if phase=='Train':
                        optimizer.zero_grad()
                        scaler.scale(loss_final).backward()
                        scaler.step(optimizer)
                        scaler.update()

                    with torch.no_grad():
                        Precision_res = Precision_func(p_main, y)
                        Recall_res = Recall_func(p_main, y)
                        DiceScore_res = DiceScore_func(p_main, y)
                        specificity_res = Specificity_func(p_main, y)
                        ACC_res = ACC_func(torch.sigmoid(p_main), y)

                        rate_batch[k, 0] = Precision_res
                        rate_batch[k, 1] = Recall_res
                        rate_batch[k, 2] = DiceScore_res
                        rate_batch[k, 3] = specificity_res
                        rate_batch[k, 4] = ACC_res

            loss_epoch[epoch, i] = torch.mean(loss_batch.cpu(), axis=0)
            rate_epoch[epoch, i] = torch.mean(rate_batch.cpu(), axis=0)
        
        if scheduler is not None: scheduler.step()
        
        es_loss = es(loss_epoch, epoch, inverse=False)
        es_rate = es(rate_epoch, epoch, inverse=True)

        if epoch==1:
            logging.info(f'===== Loss Monitoring =====')
            logging.info(f'Loss: {loss_keys}, Rate: {rate_keys} (Train, Valid)')
            
        if epoch%hp.monitoring_cycle==0:
            msg = f'{epoch:5.0f}/{hp.epochs:5.0f} '
            for l in range(len(loss_keys)):
                loss_train = loss_epoch[epoch, 0, l]
                loss_valid = loss_epoch[epoch, 1, l]
                loss_ratio = (loss_train/loss_valid)*100
                msg += f'({loss_train:6.4f}, {loss_valid:6.4f}) {es_loss[l]} '
            msg += '* '
            for l in range(len(rate_keys)):
                rate_train = rate_epoch[epoch, 0, l]
                rate_valid = rate_epoch[epoch, 1, l]
                rate_ratio = (rate_train/rate_valid)*100
                msg += f'({rate_train:6.4f}, {rate_valid:6.4f}) {es_rate[l]} '
            logging.info(msg)
            
            # Save
            history = {'loss':loss_epoch, 
                      'rate':rate_epoch, 
                      'loss_keys':loss_keys, 
                      'rate_keys':rate_keys}
            
            make_dir(f'{hp.path_model}/fold_{idx}')
            torch.save(history, f'{hp.path_model}/fold_{idx}/history.pt')

            if epoch%hp.save_cycle==0:
                torch.save(model.state_dict(), f'{hp.path_model}/fold_{idx}/model_{epoch}.pt')
                last_point = {'epoch':epoch, 
                              'history':history, 
                              'model':model.state_dict(), 
                              'optimizer':optimizer.state_dict(), 
                              'scaler':scaler.state_dict(), 
                              'scheduler':scheduler.state_dict()}
                torch.save(last_point, f'{hp.path_model}/fold_{idx}/last_point.pt')

        # best 모델 저장 - validation GUL loss 기준 (낮을수록 좋음)
        if loss_epoch[epoch, 1, 3] < best_metric:  
            best_metric = loss_epoch[epoch, 1, 3].item()
            torch.save(model.state_dict(), f'{hp.path_model}/fold_{idx}/best.pt')

    print('last epoch inference')
    inference(hp, test_loader=test_loader, epoch=hp.epochs, clear=False, visualize=False, num_fold=idx)
    print('best epoch inference')
    inference(hp, test_loader=test_loader, epoch='best', clear=False, visualize=False, num_fold=idx)


In [ ]:
inference(hp,test_loader = test_loader, epoch='best', clear=False, visualize=True, num_fold = 0)